# Feature Track 1: Synthetic Dataset Generation

While the baseline evaluated our RAG pipeline against manual queries, a production-grade system requires a **Validation Dataset**.

Manually writing these is slow and subject to human bias. In this notebook, we use **LLM-assisted Synthetic Data Generation** to create a robust evaluation suite directly from our knowledge base.

### The Goal
To generate a diverse set of `(Query, Context, Ground Truth)` triplets that specifically target the failure modes identified in the baseline:
* **Hallucinations** regarding non-existent products.
* **Temporal Conflicts** between old and new EPD figures.
* **Compliance Risks** regarding unverified sustainability claims.

### The Generation Workflow
1.  **Manual samples:** Define a small, high-quality reference set of hand-crafted `(query, reference)` pairs targeting known failure modes.
2.  **Source Sampling:** Extract high-quality chunks from our existing `ChromaDB` vector store.
3.  **LLM Generation:** Use an LLM to generate a `(query, reference answer)` pair grounded strictly in each sampled chunk.
4.  **Export:** Save the dataset in a format compatible with RAGAS for automated testing.

In [ ]:
import os
import json
import random
import pathlib
import warnings
from dotenv import load_dotenv

from sme_kt_zh_collaboration_rag.feature0_baseline_rag import DATA_DIR
from conversational_toolkit.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
)
from conversational_toolkit.embeddings.openai import (
    OpenAIEmbeddings,
)
from conversational_toolkit.vectorstores.chromadb import ChromaDBVectorStore
from conversational_toolkit.llms.base import Roles, MessageContent
from conversational_toolkit.llms.base import LLMMessage

from sme_kt_zh_collaboration_rag.feature0_baseline_rag import (
    EMBEDDING_MODEL,
    VS_PATH,
    build_llm,
    load_chunks,
    build_vector_store,
)

load_dotenv(dotenv_path="../../.env.local")
warnings.filterwarnings("ignore", category=DeprecationWarning)

_secret_path = pathlib.Path("/secrets/OPENAI_API_KEY")
if "OPENAI_API_KEY" not in os.environ and _secret_path.exists():
    os.environ["OPENAI_API_KEY"] = _secret_path.read_text().strip()

RETRIEVER_TOP_K = 5
BACKEND = "openai"  # "ollama" or "openai"
FORCE_REBUILD = False  # set True to re-embed from scratch and rebuild the vector store

if not BACKEND:
    raise ValueError('Set BACKEND to "ollama" or "openai" before running.')

## Phase 1: Manual "Golden Query" Creation

Before automating with synthetic data, we must establish a small, high-quality **Reference Set**. These manual queries serve as the "True North" for our RAG system, representing the most critical questions a user might ask.

### Why Manual Queries Still Matter
While synthetic data provides scale, manual queries capture business intent. They allow us to:
* **Target known failure modes:** Explicitly test if the system still hallucinates "Lara Pallet."
* **Verify nuance:** Test if the system can distinguish between a "verified EPD" and a "self-declared claim."
* **Benchmark the LLM Judge:** We use these to see if RAGAS scores align with our human intuition.

### The "Ground Truth" Requirement
For every manual query, we must provide:
1. **The Query:** The exact string the user would type.
2. **The Category:** The failure mode or intent it tests (e.g., "hallucination", "temporal_conflict").
2. **The Reference Context:** The specific document IDs or text snippets that *should* have been retrieved.
3. **The Ground Truth:** The perfect, concise answer the LLM should have generated.

---

In the cell below, we will define a list of dictionaries containing our manual validation samples to be used alongside the synthetic ones.

In [ ]:
manual_samples = [
    {
        "id": "1",
        "category": "portfolio_scope",
        "user_input": 'Does PrimePack AG offer a product called the "Lara Pallet"?',
        "reference": "No. The Lara Pallet is not part of PrimePack AG's portfolio. The product catalog explicitly lists it under products that are not offered. The active pallet portfolio consists of: Noé Pallet (32-100), Wooden Pallet 1208 (32-101), Recycled Plastic Pallet (32-102), Logypal 1 (32-103), LogyLight (32-104), and EP 08 (32-105). Customers should be referred to the current product catalog.",
        "sources": ["data/artificial_markdown/ART_product_catalog.pdf"],
    },
    {
        "id": "2",
        "category": "claim_verification",
        "user_input": "Can the 68% CO₂ reduction claim for the tesapack ECO & ULTRA STRONG ecoLogo (product 50-102) be included in a customer sustainability response?",
        "reference": "No, not as a stated fact. It is an internal assessment by Tesa SE (Level B/C evidence) and lacks third-party LCA/EPD verification. Policy requires Level A (verified EPD) for customer-facing facts. It may only be mentioned with the caveat: 'self-declared by Tesa SE, not independently verified.' Carbon neutrality targets for 2025 must be labeled as forward-looking goals.",
        "sources": [
            "data/artificial_markdown/ART_supplier_brochure_tesa_ECO.pdf",
            "data/artificial_markdown/ART_internal_procurement_policy.pdf",
        ],
    },
    {
        "id": "3",
        "category": "missing_data",
        "user_input": "What verified environmental data is available for the LogyLight pallet (product 32-104)?",
        "reference": "None. The datasheet states GWP and impact data are 'not yet available'. While an LCA (REL-LCA-2024-07) is commissioned, no verified figures exist. The 75% recycled content is a self-declaration, not an audit. It must not be used in customer-facing comparisons until an EPD is published.",
        "sources": ["data/artificial_markdown/ART_logylight_incomplete_datasheet.pdf"],
    },
    {
        "id": "4",
        "category": "missing_data",
        "user_input": "Are any of PrimePack AG's tape products confirmed to be PFAS-free?",
        "reference": "No. As of January 2025, no PFAS declarations have been received from IPG or Tesa SE. This is an open non-compliance item. The mention of 'solvent-free' adhesives does not equal a PFAS-free declaration. No tape may be described as PFAS-free yet.",
        "sources": [
            "data/artificial_markdown/ART_internal_procurement_policy.pdf",
            "data/ART_response_inquiry_frische_felder.pdf",
        ],
    },
    {
        "id": "5",
        "category": "source_conflict",
        "user_input": "Which GWP figure should be cited for the Relicyc Logypal 1 pallet (product 32-103), and why?",
        "reference": "The 2023 third-party verified EPD (No. S-P-10482) is the authoritative source. The 2021 datasheet citing 4.1 kg CO₂e is SUPERSEDED and used outdated methodology. Policy requires preferring the most recent third-party verified source and flagging the conflict.",
        "sources": [
            "data/EPD_pallet_relicyc_logypal1.pdf",
            "data/artificial_markdown/ART_relicyc_logypal1_datasheet_2021.pdf",
            "data/artificial_markdown/ART_internal_procurement_policy.pdf",
        ],
    },
    {
        "id": "6",
        "category": "direct_fact",
        "user_input": "What is the total 'Climate Change - total' GWP figure for the IPG F4090-05 machine roll tape per square meter?",
        "reference": "The total Climate Change GWP for the IPG F4090-05 machine roll is 2.03E-01 kg CO2 eq. per m2. This is comprised of 1.29E-01 kg CO2 eq. from the upstream stage, 6.69E-02 kg CO2 eq. from the core stage, and 7.50E-03 kg CO2 eq. from the downstream stage.",
        "sources": ["data/EPD_tape_IPG_hotmelt.pdf"],
    },
    {
        "id": "7",
        "category": "direct_fact",
        "user_input": "What is the weight and static load capacity of the Stabilplastik EP 08 pallet?",
        "reference": "The Stabilplastik EP 08 pallet has a weight of 25 kg and a static load capacity of 10,000 kg. It is designed for rigorous industrial use and is resistant to moisture, pests, and mold.",
        "sources": ["data/EPD_pallet_stabilplastik_ep08.pdf"],
    },
    {
        "id": "8",
        "category": "direct_fact",
        "user_input": "How does the EU Commission define 'Transition Risks' in the context of climate reporting?",
        "reference": "Transition risks are defined as risks to a company arising from the transition to a low-carbon and climate-resilient economy. These include policy risks (e.g., carbon-pricing), legal risks (litigation), technology risks (replacement by cleaner tech), market risks (shifting consumer choices), and reputational risks.",
        "sources": ["data/REF_eu_climate_reporting_guidelines.pdf"],
    },
    {
        "id": "9",
        "category": "multi_hop",
        "user_input": "Does the Stabilplastik EP 08 pallet meet the 2025 carbon neutrality target requirements for customer sustainability responses?",
        "reference": "The Stabilplastik EP 08 has a third-party verified EPD valid until 2028, meeting the policy's requirement for verified facts (Level A evidence). however, any claims regarding 'carbon neutrality' must still be labeled as forward-looking goals per the internal policy, as the EPD itself focuses on life-cycle impacts rather than a neutrality guarantee.",
        "sources": [
            "data/EPD_pallet_stabilplastik_ep08.pdf",
            "data/artificial_markdown/ART_internal_procurement_policy.pdf",
        ],
    },
    {
        "id": "10",
        "category": "direct_fact",
        "user_input": "What is the primary material source for the Relicyc Logypal 1 according to its 2021 datasheet?",
        "reference": "The Logypal 1 is manufactured from 100% post-consumer recycled plastic. This material is primarily sourced from end-of-life agricultural packaging (such as silage film) and industrial packaging waste.",
        "sources": ["data/artificial_markdown/ART_relicyc_logypal1_datasheet_2021.pdf"],
    },
]

## Phase 2: Synthetic Data Generation (Source Sampling)

To scale our validation set, we use the documents already stored in our Vector DB. This process involves:

1. **Random Sampling**: Selecting diverse chunks from the vector store to ensure we cover various products and policies.
2. **Context Injection**: Providing these chunks to a "Generator LLM" to draft realistic user queries.
3. **Answer Synthesis**: Using an "Oracle LLM" to write the ground truth based strictly on the provided text.

This ensures that our evaluation isn't just testing the LLM's general knowledge, but its ability to retrieve and reason over *our specific* proprietary data.

In [ ]:
# Build (or reuse) the vector store
if "sentence-transformers" in EMBEDDING_MODEL:
    embedding_model = SentenceTransformerEmbeddings(model_name=EMBEDDING_MODEL)
else:
    embedding_model = OpenAIEmbeddings(model_name=EMBEDDING_MODEL)

if FORCE_REBUILD or not VS_PATH.exists():
    chunks = load_chunks()
    vs = await build_vector_store(
        chunks, embedding_model, db_path=VS_PATH, reset=FORCE_REBUILD
    )
else:
    # Vector store already exists -> open it without re-embedding
    vs = ChromaDBVectorStore(db_path=str(VS_PATH))
    print(
        f"Reusing existing vector store at {VS_PATH} ({vs.collection.count()} chunks)"
    )

llm = build_llm(backend=BACKEND)

print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Vector store: {VS_PATH}")
print(f"LLM backend: {BACKEND}")
print("Setup complete.")

In [ ]:
# 1. Fetch data from the existing vector store. We get the raw text (documents) and metadata (sources)
data = vs.collection.get(include=["documents", "metadatas"])

# 2. Organize into a list of potential contexts
all_chunks = [
    {"content": doc, "source": meta.get("source", "unknown")}
    for doc, meta in zip(data["documents"] or [], data["metadatas"] or [])
]

# 3. Sample a subset to generate questions from (e.g., 10 chunks)
SAMPLE_SIZE = 10
sampled_chunks = random.sample(all_chunks, min(SAMPLE_SIZE, len(all_chunks)))

# Display the first sampled chunk for verification
print(f"Sampled {len(sampled_chunks)} chunks for generation.")
print("-" * 30)
print(f"Source: {sampled_chunks[0]['source']}")
print(f"Content preview: {sampled_chunks[0]['content'][:200]}...")

---

## Phase 3: LLM Generation

For each sampled chunk, we prompt the LLM to generate a realistic `(query, reference answer)` pair grounded strictly in the provided context. The model is instructed to produce questions a procurement officer or sustainability auditor would ask, with answers derived only from the chunk text.

In [ ]:
GENERATOR_PROMPT = """
Your task is to create a high-quality Question and Answer pair based STRICTLY on the provided Context.

Context:
{context}

Guidelines:
1. The Question should be something a procurement officer or sustainability auditor would ask.
2. The Answer must be factual, concise, and derived ONLY from the Context.
3. If the Context mentions a specific product, ID, or date, include it in the question or answer.
4. Format your response as:
QUESTION: <question>
ANSWER: <answer>
"""

synthetic_samples = []

for i, chunk in enumerate(sampled_chunks):
    prompt = GENERATOR_PROMPT.format(context=chunk["content"])

    # Generate the Q&A pair using the LLM
    response = await llm.generate(
        [
            LLMMessage(
                role=Roles.SYSTEM, content=[MessageContent(type="text", text=prompt)]
            )
        ]
    )
    try:
        # Simple parsing logic
        content = response.content[0].text
        if content is None:
            raise ValueError("LLM returned None content")
        parts = content.split("ANSWER:")
        question = parts[0].replace("QUESTION:", "").strip()
        answer = parts[1].strip()

        sample = {
            "user_input": question,
            "reference": answer,
            "sources": [chunk["source"]],
        }
        synthetic_samples.append(sample)
        print(f"[{i + 1}/{len(sampled_chunks)}] Generated: {question[:50]}...")
    except Exception as e:
        print(f"[{i + 1}/{len(sampled_chunks)}] Failed to parse response: {e}")

print(f"\nTotal Synthetic Samples: {len(synthetic_samples)}")

In [ ]:
print("Sample Synthetic Q&A Pair:")
print("-" * 30)
print(f"user_input: {synthetic_samples[1]['user_input']}")
print(f"reference: {synthetic_samples[1]['reference']}")
print(f"sources: {synthetic_samples[1]['sources']}")

---

## Phase 4: Export

Combine the manual and synthetic samples and save to `evaluation_dataset.json` for use in automated RAGAS evaluation.

In [ ]:
all_samples = manual_samples + synthetic_samples
output_path = DATA_DIR / "evaluation_dataset.json"
output_path.write_text(json.dumps(all_samples, indent=2, ensure_ascii=False))
print(
    f"Saved {len(all_samples)} samples ({len(manual_samples)} manual + {len(synthetic_samples)} synthetic) to {output_path}"
)